In [1]:
#pip install bert-score

In [ ]:
import pandas as pd
import re
from pathlib import Path
from bert_score import BERTScorer

csv_path = Path("/home/vmadmin/intent/src/test/anthropic-opus-4-5.csv")
raw_df = pd.read_csv(csv_path)

# Extract JSON-like tool arguments from the serialized response blob
def extract_arguments(payload):
    if not isinstance(payload, str):
        return None
    match = re.search(r"input=(.*?),\s*name=", payload, flags=re.DOTALL)
    args = match.group(1) if match else None
    if not match:
        return None
    return match.group(1).replace('""', '"')

intent_arguments_df = raw_df.assign(
    arguments=raw_df["intent_processing"].map(extract_arguments)
)[["intent", "arguments"]]

intent_arguments_df

,intent,arguments
0,Create a slice to support video journalists tr...,"{'body': {'serviceTime': None, 'serviceArea': ..."
1,Provision a slice for a university campus even...,"{'body': {'serviceTime': None, 'serviceArea': ..."
2,Set up a slice for a shopping mall to improve ...,"{'body': {'serviceTime': None, 'serviceArea': ..."
3,Deploy a slice for a media production crew usi...,"{'body': {'serviceTime': None, 'serviceArea': ..."
4,Establish a slice for commuters in a busy trai...,"{'body': {'serviceTime': None, 'serviceArea': ..."
...,...,...
96,Provision a slice for smart greenhouse climate...,"{'body': {'serviceTime': None, 'serviceArea': ..."
97,Establish a slice for public safety siren moni...,"{'body': {'serviceTime': None, 'serviceArea': ..."
98,Deploy a slice for high-resolution mobile mapp...,"{'body': {'serviceTime': None, 'serviceArea': ..."
99,Create a slice for autonomous ferry navigation...,"{'body': {'serviceTime': None, 'serviceArea': ..."


In [8]:
intent_arguments_df = intent_arguments_df.head(10)
intent_arguments_df

,intent,arguments
0,Create a slice to support video journalists tr...,"{'body': {'serviceTime': None, 'serviceArea': ..."
1,Provision a slice for a university campus even...,"{'body': {'serviceTime': None, 'serviceArea': ..."
2,Set up a slice for a shopping mall to improve ...,"{'body': {'serviceTime': None, 'serviceArea': ..."
3,Deploy a slice for a media production crew usi...,"{'body': {'serviceTime': None, 'serviceArea': ..."
4,Establish a slice for commuters in a busy trai...,"{'body': {'serviceTime': None, 'serviceArea': ..."
5,Create a slice for a sports arena during live ...,"{'body': {'serviceTime': None, 'serviceArea': ..."
6,Provision a slice for an outdoor music festiva...,"{'body': {'serviceTime': None, 'serviceArea': ..."
7,Design a slice for a business district to supp...,"{'body': {'serviceTime': None, 'serviceArea': ..."
8,Deploy a slice for a convention center hosting...,"{'body': {'serviceTime': None, 'serviceArea': ..."
9,Create a slice to enhance mobile broadband alo...,"{'body': {'serviceTime': None, 'serviceArea': ..."


In [12]:
scorer = BERTScorer(
    model_type="microsoft/deberta-xlarge-mnli",
    lang="en",
    use_fast_tokenizer=True,
)
scorer._tokenizer.model_max_length = 512
scorer._tokenizer.init_kwargs["model_max_length"] = 512  # keep future loads consistent


def score_pair(row: pd.Series) -> pd.Series:
    intent = row.intent
    arguments = row.arguments

    if not isinstance(intent, str) or not isinstance(arguments, str):
        return pd.Series({"precision": None, "recall": None, "F1 Score": None})

    # BERTScorer expects lists of strings, not raw strings.
    P, R, F1 = scorer.score([intent], [arguments])

    return pd.Series(
        {
            "precision": float(P.mean()),
            "recall": float(R.mean()),
            "F1 Score": float(F1.mean()),
        }
    )

intent_arguments_df = intent_arguments_df.join(
    intent_arguments_df.apply(score_pair, axis=1)
)

intent_arguments_df

,intent,arguments,precision,recall,F1 Score
0,Create a slice to support video journalists tr...,"{""body"":{""serviceTime"":null,""serviceArea"":null...",0.491074,0.379803,0.428330
1,Provision a slice for a university campus even...,"{""body"":{""serviceTime"":null,""serviceArea"":null...",0.484881,0.408464,0.443404
2,Set up a slice for a shopping mall to improve ...,"{""body"":{""serviceTime"":null,""serviceArea"":null...",0.475040,0.382498,0.423775
3,Deploy a slice for a media production crew usi...,"{""body"":{""serviceTime"":null,""serviceArea"":null...",0.462025,0.356579,0.402511
4,Establish a slice for commuters in a busy trai...,"{""body"":{""serviceTime"":null,""serviceArea"":null...",0.512264,0.364474,0.425912
5,Create a slice for a sports arena during live ...,"{""body"":{""serviceTime"":null,""serviceArea"":null...",0.476434,0.352897,0.405464
6,Provision a slice for an outdoor music festiva...,"{""body"":{""serviceTime"":null,""serviceArea"":null...",0.481657,0.367080,0.416635
7,Design a slice for a business district to supp...,"{""body"":{""serviceTime"":null,""serviceArea"":null...",0.504950,0.340244,0.406548
8,Deploy a slice for a convention center hosting...,"{""body"":{""serviceTime"":null,""serviceArea"":null...",0.500192,0.363140,0.420787
9,Create a slice to enhance mobile broadband alo...,"{""body"":{""serviceTime"":null,""serviceArea"":null...",0.485864,0.346718,0.404663


In [17]:
intent_arguments_df.to_csv("/home/vmadmin/intent/src/results/bert_score_test.csv", index=False, header=True)